# Les 08 - Multi-Agent Ontwerppatroon


## Installatie


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Waarom Multi-Agent Systemen?

Taken in de echte wereld, zoals reisplanning, vereisen vele verschillende soorten expertise — logistiek, lokale kennis, budgettering en meer. Een enkele agent die alles probeert te doen wordt snel onhandelbaar.

Multi-agent systemen lossen dit op door **specialisatie**: elke agent richt zich op een specifiek expertisegebied, wat resulteert in hogere kwaliteit dan een generalist. Ze verbeteren ook de **schaalbaarheid** — je kunt nieuwe agenten toevoegen (bijv. een vliegspecialist, een restaurantcriticus) zonder de bestaande workflow te herschrijven. De agenten werken samen via een gestructureerde pijplijn, waarbij context van de ene naar de andere wordt doorgegeven.


## Speciale Agenten Maken


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Een sequentiële workflow bouwen

`WorkflowBuilder` stelt je in staat om agenten in een gerichte grafiek met elkaar te verbinden. Hier maken we een eenvoudige twee-stappen pijplijn: de **TravelPlanner** stelt de route op, daarna bekijkt en verbetert de **TravelConcierge** deze.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Meer agenten toevoegen aan de workflow

Een van de grootste voordelen van het multi-agentenpatroon is hoe gemakkelijk het is uit te breiden. Hieronder voegen we een **BudgetReviewer** agent toe die het plan controleert aan de hand van het budget van de reiziger, items markeert die de kosten boven het limiet kunnen duwen, en geldbesparende alternatieven suggereert. De workflow voert nu drie agenten achtereenvolgens uit:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Samenvatting

In deze les heb je geleerd hoe je:

1. **Gespecialiseerde agenten maakt** — elk met een specifieke rol (planning, conciërge, budgetbeoordeling).
2. **Agenten aansluit in een opeenvolgende workflow** met `WorkflowBuilder` en `add_edge`.
3. **Output streamt** van een multi-agent pijplijn, waarbij wordt bijgehouden welke agent spreekt.
4. **Een workflow uitbreidt** door nieuwe agenten toe te voegen aan de keten zonder bestaande te wijzigen.

Het multi-agent ontwerp patroon houdt elke agent eenvoudig terwijl het rijkere, grondiger beoordeelde resultaten produceert dan één enkele agent zou kunnen bereiken.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
